In [1]:
import numpy as np
import pandas as pd
import scipy

from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, log_evaluation, early_stopping
from xgboost import XGBClassifier

from warnings import filterwarnings

filterwarnings('ignore')

train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/test.csv')

In [2]:
def add_features(df):
    df = df.copy()

    df["u_g"] = df["u"] - df["g"]
    df["g_r"] = df["g"] - df["r"]
    df["r_i"] = df["r"] - df["i"]
    df["i_z"] = df["i"] - df["z"]
    
    df["u_r"] = df["u"] - df["r"]
    df["u_i"] = df["u"] - df["i"]
    df["u_z"] = df["u"] - df["z"]
    
    df["g_i"] = df["g"] - df["i"]
    df["g_z"] = df["g"] - df["z"]
    
    df["r_z"] = df["r"] - df["z"]
    
    bands = ["u", "g", "r", "i", "z"]
    
    df["mag_mean"] = df[bands].mean(axis=1)
    df["mag_std"] = df[bands].std(axis=1)
    df["mag_min"] = df[bands].min(axis=1)
    df["mag_max"] = df[bands].max(axis=1)
    df["mag_range"] = df["mag_max"] - df["mag_min"]
    
    df["redshift_u"] = df["redshift"] * df["u"]
    df["redshift_g"] = df["redshift"] * df["g"]
    df["redshift_r"] = df["redshift"] * df["r"]
    df["redshift_i"] = df["redshift"] * df["i"]
    df["redshift_z"] = df["redshift"] * df["z"]
    
    df["alpha_rad"] = np.deg2rad(df["alpha"])
    
    df["alpha_sin"] = np.sin(df["alpha_rad"])
    df["alpha_cos"] = np.cos(df["alpha_rad"])
    
    df["delta_rad"] = np.deg2rad(df["delta"])
    
    df["delta_sin"] = np.sin(df["delta_rad"])
    df["delta_cos"] = np.cos(df["delta_rad"])
    
    spectral_map = {
        "O/B": 0,
        "A": 1,
        "F": 2,
        "G": 3,
        "K": 4,
        "M": 5,
    }
    
    df["spectral_ord"] = df["spectral_type"].map(spectral_map)
    df = df.drop(columns=["alpha_rad", "delta_rad"])

    for band in bands:
        df[f"flux_{band}"] = 10 ** (-0.4 * df[band])

    flux_cols = [f"flux_{b}" for b in bands]

    df["flux_mean"] = df[flux_cols].mean(axis=1)
    df["flux_std"] = df[flux_cols].std(axis=1)
    df["flux_min"] = df[flux_cols].min(axis=1)
    df["flux_max"] = df[flux_cols].max(axis=1)
    df["flux_range"] = df["flux_max"] - df["flux_min"]

    x = np.arange(len(bands))
    x_centered = x - x.mean()
    denom = np.sum(x_centered ** 2)
    
    df["mag_slope"] = (
        df[bands]
        .sub(df[bands].mean(axis=1), axis=0)
        .dot(x_centered)
        / denom
    )

    df["mag_curvature"] = (
        df["u"] 
        - 2 * df["r"] 
        + df["z"]
    )
    df["blue_curvature"] = df["u"] - 2 * df["g"] + df["r"]
    df["red_curvature"] = df["r"] - 2 * df["i"] + df["z"]

    eps = 1e-6

    df["u_g_per_redshift"] = df["u_g"] / (df["redshift"].abs() + eps)
    df["g_r_per_redshift"] = df["g_r"] / (df["redshift"].abs() + eps)
    df["r_i_per_redshift"] = df["r_i"] / (df["redshift"].abs() + eps)
    df["i_z_per_redshift"] = df["i_z"] / (df["redshift"].abs() + eps)

    return df

train_fe = add_features(train)
test_fe = add_features(test)

In [3]:
TARGET = "class"
ID_COL = "id"

cat_cols = ["spectral_type", "galaxy_population"]

features = [c for c in train_fe.columns if c not in [ID_COL, TARGET]]

X = train_fe[features].copy()
X_test = test_fe[features].copy()
y = train_fe[TARGET].copy()

le = LabelEncoder()
y_enc = le.fit_transform(y)

n_classes = len(le.classes_)

In [4]:
# For LightGBM
X_lgb = X.copy()
X_test_lgb = X_test.copy()

for col in cat_cols:
    X_lgb[col] = X_lgb[col].astype("category")
    X_test_lgb[col] = X_test_lgb[col].astype("category")

# For XGBoost: one-hot encoding
X_xgb = pd.get_dummies(X, columns=cat_cols)
X_test_xgb = pd.get_dummies(X_test, columns=cat_cols)

X_test_xgb = X_test_xgb.reindex(columns=X_xgb.columns, fill_value=0)

In [5]:
# https://www.kaggle.com/code/emanuellcs/predicting-stellar-class-xgboost
xgb_params = {
    'objective': 'multi:softprob',
    'num_class': 3,
    'eval_metric': 'mlogloss',
    'tree_method': 'hist',           
    
    'learning_rate': 0.015,
    'n_estimators': 5000,
    'early_stopping_rounds': 150,
    
    'max_depth': 8,
    'min_child_weight': 5,
    'max_delta_step': 1,
    
    'gamma': 0.2,
    'reg_alpha': 0.5,
    'reg_lambda': 2.5,
    
    'subsample': 0.75,
    'colsample_bytree': 0.7,
    'colsample_bylevel': 0.8,
    
    'enable_categorical': True,
    'random_state': 42,

    'device': 'cuda'
}

In [6]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_cat = np.zeros((len(X), n_classes))
oof_lgb = np.zeros((len(X), n_classes))
oof_xgb = np.zeros((len(X), n_classes))

test_cat = np.zeros((len(X_test), n_classes))
test_lgb = np.zeros((len(X_test), n_classes))
test_xgb = np.zeros((len(X_test), n_classes))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y_enc)):
    print(f"\nFold {fold + 1}")

    y_train = y_enc[train_idx]
    y_valid = y_enc[valid_idx]

    sample_weight = compute_sample_weight(
        class_weight="balanced",
        y=y_train
    )

    # CatBoost
    cat_model = CatBoostClassifier(
        loss_function="MultiClass",
        iterations=5000,
        learning_rate=0.03,
        depth=10,
        random_seed=42,
        task_type="GPU",
        devices="0:1",
        verbose=1000
    )

    cat_model.fit(
        X.iloc[train_idx],
        y_train,
        cat_features=cat_cols,
        sample_weight=sample_weight,
        eval_set=(X.iloc[valid_idx], y_valid),
        use_best_model=True
    )

    oof_cat[valid_idx] = cat_model.predict_proba(X.iloc[valid_idx])
    test_cat += cat_model.predict_proba(X_test) / skf.n_splits

    cat_score = balanced_accuracy_score(
        y_valid,
        np.argmax(oof_cat[valid_idx], axis=1)
    )

    print(f"CatBoost BAL ACC: {cat_score:.6f}")

    # LightGBM
    lgb_model = LGBMClassifier(
        objective="multiclass",
        num_class=n_classes,
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=-1,
        num_leaves=128,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
        device="gpu"
    )

    lgb_model.fit(
        X_lgb.iloc[train_idx],
        y_train,
        sample_weight=sample_weight,
        eval_set=[(X_lgb.iloc[valid_idx], y_valid)],
        eval_metric="multi_logloss",
        categorical_feature=cat_cols,
        callbacks=[
            log_evaluation(period=1000),
            early_stopping(stopping_rounds=200)
        ]
    )

    oof_lgb[valid_idx] = lgb_model.predict_proba(X_lgb.iloc[valid_idx])
    test_lgb += lgb_model.predict_proba(X_test_lgb) / skf.n_splits

    lgb_score = balanced_accuracy_score(
        y_valid,
        np.argmax(oof_lgb[valid_idx], axis=1)
    )

    print(f"LightGBM BAL ACC: {lgb_score:.6f}")

    # XGBoost
    xgb_model = XGBClassifier(
        **xgb_params,
        n_jobs=-1
    )

    xgb_model.fit(
        X_xgb.iloc[train_idx],
        y_train,
        sample_weight=sample_weight,
        eval_set=[(X_xgb.iloc[valid_idx], y_valid)],
        verbose=1000
    )

    oof_xgb[valid_idx] = xgb_model.predict_proba(X_xgb.iloc[valid_idx])
    test_xgb += xgb_model.predict_proba(X_test_xgb) / skf.n_splits

    xgb_score = balanced_accuracy_score(
        y_valid,
        np.argmax(oof_xgb[valid_idx], axis=1)
    )

    print(f"XGBoost BAL ACC: {xgb_score:.6f}")


Fold 1
0:	learn: 1.0487419	test: 1.0496160	best: 1.0496160 (0)	total: 181ms	remaining: 15m 6s
1000:	learn: 0.0883479	test: 0.1160669	best: 0.1160669 (1000)	total: 51s	remaining: 3m 23s
2000:	learn: 0.0718722	test: 0.1100887	best: 0.1100887 (2000)	total: 1m 43s	remaining: 2m 34s
3000:	learn: 0.0607056	test: 0.1068776	best: 0.1068771 (2998)	total: 2m 35s	remaining: 1m 43s
4000:	learn: 0.0522752	test: 0.1048209	best: 0.1048209 (4000)	total: 3m 27s	remaining: 51.7s
4999:	learn: 0.0455220	test: 0.1034929	best: 0.1034927 (4998)	total: 4m 19s	remaining: 0us
bestTest = 0.1034927227
bestIteration = 4998
Shrink model to first 4999 iterations.
CatBoost BAL ACC: 0.961626


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Training until validation scores don't improve for 200 rounds
[1000]	valid_0's multi_logloss: 0.0965686
Early stopping, best iteration is:
[1567]	valid_0's multi_logloss: 0.0955008
LightGBM BAL ACC: 0.964155
[0]	validation_0-mlogloss:1.08314
[1000]	validation_0-mlogloss:0.10762
[2000]	validation_0-mlogloss:0.10073
[3000]	validation_0-mlogloss:0.09737
[4000]	validation_0-mlogloss:0.09564
[4999]	validation_0-mlogloss:0.09488
XGBoost BAL ACC: 0.964775

Fold 2
0:	learn: 1.0487369	test: 1.0497309	best: 1.0497309 (0)	total: 90.2ms	remaining: 7m 30s
1000:	learn: 0.0877942	test: 0.1187344	best: 0.1187344 (1000)	total: 50.8s	remaining: 3m 23s
2000:	learn: 0.0713502	test: 0.1124968	best: 0.1124968 (2000)	total: 1m 42s	remaining: 2m 33s
3000:	learn: 0.0601026	test: 0.1091924	best: 0.1091924 (3000)	total: 2m 34s	remaining: 1m 42s
4000:	learn: 0.0515854	test: 0.1072209	best: 0.1072209 (4000)	total: 3m 26s	remaining: 51.6s
4999:	learn: 0.0449284	test: 0.1058677	best: 0.1058677 (4999)	total: 4m 19s	r

In [7]:
best_score = -1
best_weights = None

for w_cat in np.arange(0, 1.01, 0.05):
    for w_lgb in np.arange(0, 1.01 - w_cat, 0.05):
        w_xgb = 1.0 - w_cat - w_lgb

        blend = (
            w_cat * oof_cat +
            w_lgb * oof_lgb +
            w_xgb * oof_xgb
        )

        score = balanced_accuracy_score(
            y_enc,
            np.argmax(blend, axis=1)
        )

        if score > best_score:
            best_score = score
            best_weights = (w_cat, w_lgb, w_xgb)

print(f"Best grid score: {best_score:.6f}")
print(f"CatBoost: {best_weights[0]:.2f}")
print(f"LightGBM: {best_weights[1]:.2f}")
print(f"XGBoost : {best_weights[2]:.2f}")

Best grid score: 0.964582
CatBoost: 0.00
LightGBM: 0.45
XGBoost : 0.55


In [8]:
# Weighted OOF blend
best_oof_blend = (
    best_weights[0] * oof_cat +
    best_weights[1] * oof_lgb +
    best_weights[2] * oof_xgb
)

# Weighted test blend
best_test_blend = (
    best_weights[0] * test_cat +
    best_weights[1] * test_lgb +
    best_weights[2] * test_xgb
)

blend_score = balanced_accuracy_score(
    y_enc,
    np.argmax(best_oof_blend, axis=1)
)

print(f"Weighted blend OOF BAL ACC before multipliers: {blend_score:.6f}")

Weighted blend OOF BAL ACC before multipliers: 0.964582


In [9]:
def score_multipliers(multipliers, probs, y_true):
    multipliers = np.array(multipliers)

    adjusted_probs = probs * multipliers.reshape(1, -1)
    preds = np.argmax(adjusted_probs, axis=1)

    return -balanced_accuracy_score(y_true, preds)


initial_multipliers = np.ones(n_classes)

result = minimize(
    score_multipliers,
    initial_multipliers,
    args=(best_oof_blend, y_enc),
    method="Nelder-Mead",
    options={
        "maxiter": 2000,
        "xatol": 1e-7,
        "fatol": 1e-7
    }
)

best_multipliers = result.x
best_multipliers = best_multipliers / best_multipliers.mean()

print("Best multipliers:")
for cls, mult in zip(le.classes_, best_multipliers):
    print(f"{cls}: {mult:.6f}")

Best multipliers:
GALAXY: 0.622209
QSO: 1.089515
STAR: 1.288276


In [10]:
best_oof_blend_adjusted = (
    best_oof_blend *
    best_multipliers.reshape(1, -1)
)

adjusted_score = balanced_accuracy_score(
    y_enc,
    np.argmax(best_oof_blend_adjusted, axis=1)
)

print(f"Weighted blend OOF BAL ACC before: {blend_score:.6f}")
print(f"Weighted blend OOF BAL ACC after : {adjusted_score:.6f}")

Weighted blend OOF BAL ACC before: 0.964582
Weighted blend OOF BAL ACC after : 0.966339


In [11]:
best_test_blend_adjusted = (
    best_test_blend *
    best_multipliers.reshape(1, -1)
)

test_pred_enc = np.argmax(best_test_blend_adjusted, axis=1)
test_pred = le.inverse_transform(test_pred_enc)

submission = pd.DataFrame({
    "id": test["id"],
    "class": test_pred
})

submission.to_csv(
    "submission_oof_weighted_blend_threshold_optimized.csv",
    index=False
)

submission.head()

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY
